# Run 4 Validation

This notebook validates the output tables after Run 4 (Day 4) data load.

## Expected Changes in Run 4:
- **Customer Updates**:
  - John (ID=1): Email changed from jdoe@example.com to john.doe@another.example.com
  - DELETE_FLAG set to False (was NULL)
- **SCD2 Behavior**: 
  - John should have another version created in silver.customer


In [ ]:
%run "./initialize"


In [ ]:
# Import validation utilities
from validation_utils import ValidationRunner

# Initialize validation runner with spark session
v = ValidationRunner(spark)


## Bronze Layer Validations

Run 4 adds 1 new customer record (John's email update)


In [ ]:
print("=" * 60)
print("BRONZE LAYER - Base Samples")
print("=" * 60)

# Bronze customer - 7 from Run 3 + 1 new (John's email update)
v.validate_row_count(f"{bronze_schema}.customer", 8, "Customer CDC records (+1 John update)")

# Bronze customer_address - unchanged from Run 3
v.validate_row_count(f"{bronze_schema}.customer_address", 7, "Address CDC records (unchanged)")


## Silver Layer Validations

### SCD2 Changes:
- John (ID=1): Another version created (email changed to john.doe@another.example.com)
- Now John has 3 versions: original, jdoe, and john.doe@another.example.com


In [ ]:
print("\n" + "=" * 60)
print("SILVER LAYER - Base Samples")
print("=" * 60)

# Silver customer - still 4 active (John, Jane, Alice, Joe)
# Richard was deleted in Run 2
v.validate_active_scd2_count(f"{silver_schema}.customer", 4, "__END_AT")

# Silver customer - should now have 3 closed records:
# 1. John's original version
# 2. John's jdoe version
# 3. Richard (deleted)
v.validate_min_closed_scd2_count(f"{silver_schema}.customer", 3, "__END_AT")

# Validate John's current email is the new one
v.validate_column_value(
    f"{silver_schema}.customer",
    "CUSTOMER_ID = 1 AND __END_AT IS NULL",
    "EMAIL",
    "john.doe@another.example.com",
    "John's latest email (active record)"
)

# Customer address unchanged from Run 3
v.validate_min_closed_scd2_count(f"{silver_schema}.customer_address", 2, "__END_AT")


## Gold Layer Validations


In [ ]:
print("\n" + "=" * 60)
print("GOLD LAYER - Stream-Static")
print("=" * 60)

# TODO - Fix Gold pattern for DWH, currently does not have intended behavior
# # Gold dimension should have accumulated more history
# v.validate_min_row_count(
#     f"{gold_schema}.dim_customer_sql_sample",
#     6,  # At least 6 rows including all history
#     "Final dimension records with full history"
# )

# # Verify active record count
# v.validate_active_scd2_count(
#     f"{gold_schema}.dim_customer_sql_sample",
#     4,  # 4 active customers (John, Jane, Alice, Joe) - Richard deleted
#     "__END_AT"
# )


## Multi-Source Streaming Final State


In [ ]:
print("\n" + "=" * 60)
print("SILVER LAYER - Multi-Source Streaming Final State")
print("=" * 60)

# Multi-source streaming should have accumulated history
v.validate_active_scd2_count(
    f"{silver_schema}.customer_ms_basic",
    4,  # 4 active (John, Jane, Alice, Joe)
    "__END_AT"
)

# Should have multiple closed records from all the changes
v.validate_min_closed_scd2_count(
    f"{silver_schema}.customer_ms_basic",
    2,  # At least 2 closed (Richard deleted, plus changes)
    "__END_AT"
)

print("\n" + "=" * 60)
print("OPERATIONAL METADATA - meta_load_details Validation")
print("=" * 60)

# Validate meta_load_details nested column fields are not null
# Using wildcard to check all fields in the struct
v.validate_column_not_null(
    f"{silver_schema}.customer",
    "meta_load_details.pipeline_start_timestamp",
    "Silver customer meta_load_details"
)

v.validate_column_not_null(
    f"{silver_schema}.customer_address",
    "meta_load_details.pipeline_start_timestamp",
    "Silver customer_address meta_load_details"
)

v.validate_column_not_null(
    f"{silver_schema}.customer_ms_basic",
    "meta_load_details.pipeline_start_timestamp",
    "Silver customer_ms_basic meta_load_details"
)


## Validation Summary


In [ ]:
# Validation summary runs at the end of the notebook after all checks complete
pass

In [ ]:
print("\n" + "=" * 60)
print("BRONZE LAYER - Feature General Pipeline (Run 4)")
print("=" * 60)

# append_sql_flow: unchanged from Run 3 (no new addresses in Run 4)
v.validate_row_count(f"{bronze_schema}.append_sql_flow", 8, "Address rows after Run 4 (unchanged)")

# append_view_flow: 7 from Run 3 + 1 new (John email update) = 8
v.validate_row_count(f"{bronze_schema}.append_view_flow", 8, "Customer rows after Run 4 (+1 John update)")

# feature_constraints: 7 + 1 = 8 rows
v.validate_row_count(f"{bronze_schema}.feature_constraints", 8, "Customer rows with DDL constraints after Run 4")

# feature_version_mapping_flows: SCD1, John updated to new email
v.validate_row_count(f"{bronze_schema}.feature_version_mapping_flows", 5, "SCD1 version mapping (5 rows, John updated)")
v.validate_column_value(
    f"{bronze_schema}.feature_version_mapping_flows",
    "CUSTOMER_ID = 1",
    "EMAIL",
    "john.doe@another.example.com",
    "John's final email in SCD1 table"
)

# v_version_mapping_standard: same SCD1 behaviour
v.validate_row_count(f"{bronze_schema}.v_version_mapping_standard", 5, "SCD1 standard (5 rows after Run 4)")
v.validate_column_value(
    f"{bronze_schema}.v_version_mapping_standard",
    "CUSTOMER_ID = 1",
    "EMAIL",
    "john.doe@another.example.com",
    "John's final email in SCD1 standard"
)

print("\n" + "=" * 60)
print("BRONZE LAYER - Feature Snapshots (Additional Tables) Run 4")
print("=" * 60)

# Periodic snapshot SCD1: no snapshot source change in Run 4 (same 4 rows as Run 3)
v.validate_row_count(
    f"{bronze_schema}.feature_periodic_snapshot_scd1",
    4,
    "Periodic snapshot SCD1 (4 customers, unchanged from Run 3)"
)

print("\n" + "=" * 60)
print("BRONZE LAYER - Feature Table Migration Pipeline Run 4")
print("=" * 60)

# feature_migrated_table_scd2: Run 4 CDC does not produce additional SCD2 versions in this table
# Active: still 5 keys; Closed: still 1 (same as Run 2/3)
v.validate_active_scd2_count(
    f"{bronze_schema}.feature_migrated_table_scd2",
    5,
    "__END_AT"
)
v.validate_min_closed_scd2_count(
    f"{bronze_schema}.feature_migrated_table_scd2",
    1,
    "__END_AT"
)

# feature_migrated_table_append_only: accumulates rows from streaming CDC
v.validate_min_row_count(
    f"{bronze_schema}.feature_migrated_table_append_only",
    3,
    "Append-only migration (at least 3 rows after Run 4)"
)

print("\n" + "=" * 60)
print("BRONZE LAYER - Template Samples After Run 4")
print("=" * 60)

# customer_template: no new snapshot files in Run 4; unchanged from Run 2
# Active: 6 records (same as Run 2/3)
v.validate_active_scd2_count(
    f"{bronze_schema}.customer_template",
    6,
    "__END_AT"
)

# customer_address_template: no new address snapshot files in Run 4; unchanged
v.validate_min_row_count(
    f"{bronze_schema}.customer_address_template",
    5,
    "Template customer_address SCD2 (at least 5 rows, unchanged in Run 4)"
)

print("\n" + "=" * 60)
print("BRONZE LAYER - Materialized Views After Run 4")
print("=" * 60)

# MVs source from staging.customer (8 rows after Run 4: 7 from Run 2/3 + 1 John update)
v.validate_row_count(f"{bronze_schema}.feature_mv_source_view", 8, "MV source view (8 rows after Run 4)")
v.validate_row_count(f"{bronze_schema}.feature_mv_sql_path", 8, "MV SQL path (8 rows after Run 4)")
v.validate_row_count(f"{bronze_schema}.feature_mv_chain_mvs", 8, "Chained MV (8 rows after Run 4)")

# feature_mv_with_quarantine: no new addresses in Run 4 (unchanged from Run 3)
v.validate_row_count(f"{bronze_schema}.feature_mv_with_quarantine", 7, "MV with quarantine (7 valid, unchanged in Run 4)")

In [ ]:
v.print_summary()